In [1]:
import polars as pl

from datetime import date

from ohlc_dss_model.data import load_parquet, remove_incomplete_days, intraday_session_tagging, session_tagging

from ohlc_dss_model.features import aggregate_sessions, yang_zhang, HISTORICAL_SPEC, TODAY_SPEC

from ohlc_dss_model.utils import convert_to_timezone

In [2]:
df = load_parquet()
df = convert_to_timezone(df)
df = session_tagging(df)
df = intraday_session_tagging(df)
df = remove_incomplete_days(df)

print(df.head(5))

shape: (5, 8)
┌──────────────────┬─────────┬─────────┬─────────┬─────────┬────────┬────────────┬─────────────────┐
│ DateTime         ┆ Open    ┆ High    ┆ Low     ┆ Close   ┆ Volume ┆ Session    ┆ Intraday_Sessio │
│ ---              ┆ ---     ┆ ---     ┆ ---     ┆ ---     ┆ ---    ┆ ---        ┆ n               │
│ datetime[ns, Ame ┆ f64     ┆ f64     ┆ f64     ┆ f64     ┆ u64    ┆ date       ┆ ---             │
│ rica/New_York]   ┆         ┆         ┆         ┆         ┆        ┆            ┆ str             │
╞══════════════════╪═════════╪═════════╪═════════╪═════════╪════════╪════════════╪═════════════════╡
│ 2016-01-03       ┆ 4592.5  ┆ 4606.75 ┆ 4592.5  ┆ 4603.25 ┆ 1134   ┆ 2016-01-04 ┆ Asia            │
│ 18:00:00 EST     ┆         ┆         ┆         ┆         ┆        ┆            ┆                 │
│ 2016-01-03       ┆ 4603.0  ┆ 4603.0  ┆ 4597.5  ┆ 4600.5  ┆ 425    ┆ 2016-01-04 ┆ Asia            │
│ 18:30:00 EST     ┆         ┆         ┆         ┆         ┆        ┆        

In [3]:
features = aggregate_sessions(df)
print(features.head(5))

shape: (5, 13)
┌────────────┬──────────┬────────────┬─────────┬───┬─────────┬──────────┬────────────┬─────────┐
│ Session    ┆ O_London ┆ O_New York ┆ O_Asia  ┆ … ┆ L_Asia  ┆ C_London ┆ C_New York ┆ C_Asia  │
│ ---        ┆ ---      ┆ ---        ┆ ---     ┆   ┆ ---     ┆ ---      ┆ ---        ┆ ---     │
│ date       ┆ f64      ┆ f64        ┆ f64     ┆   ┆ f64     ┆ f64      ┆ f64        ┆ f64     │
╞════════════╪══════════╪════════════╪═════════╪═══╪═════════╪══════════╪════════════╪═════════╡
│ 2016-01-04 ┆ 4529.75  ┆ 4498.75    ┆ 4592.5  ┆ … ┆ 4522.25 ┆ 4498.25  ┆ 4506.25    ┆ 4529.5  │
│ 2016-01-05 ┆ 4509.0   ┆ 4494.75    ┆ 4505.5  ┆ … ┆ 4498.25 ┆ 4494.75  ┆ 4481.5     ┆ 4509.25 │
│ 2016-01-06 ┆ 4445.0   ┆ 4396.25    ┆ 4482.0  ┆ … ┆ 4425.5  ┆ 4396.25  ┆ 4448.5     ┆ 4444.5  │
│ 2016-01-07 ┆ 4365.0   ┆ 4316.0     ┆ 4450.25 ┆ … ┆ 4351.5  ┆ 4316.75  ┆ 4293.25    ┆ 4365.0  │
│ 2016-01-08 ┆ 4339.75  ┆ 4337.0     ┆ 4298.0  ┆ … ┆ 4281.0  ┆ 4335.5   ┆ 4264.5     ┆ 4340.0  │
└────────────┴─

***

### **Defining Session Reference Open**
***
Reference open represents daily candle opening price and are defined as such:

$O_{Ref}(t)$ (`O_Ref_t`) = `O_Asia_t` -> For day `t`

Every M level bands will be placed relative to this level.

In [4]:
features = features.with_columns(
    pl.col("O_Asia").alias("O_Ref")
)
print(features.select(["Session", "O_Ref", "O_Asia"]).head(3))

shape: (3, 3)
┌────────────┬────────┬────────┐
│ Session    ┆ O_Ref  ┆ O_Asia │
│ ---        ┆ ---    ┆ ---    │
│ date       ┆ f64    ┆ f64    │
╞════════════╪════════╪════════╡
│ 2016-01-04 ┆ 4592.5 ┆ 4592.5 │
│ 2016-01-05 ┆ 4505.5 ┆ 4505.5 │
│ 2016-01-06 ┆ 4482.0 ┆ 4482.0 │
└────────────┴────────┴────────┘


***

### **Yang-Zhang Volatility**
***
We will be using Yang-Zhang volatility to help us predict the statistical levels, this levels will be used to capture intraday price movement.

You can use any volatility estimator but as previously stated in `02_feature_selection.ipynb` Yang-Zhang volatility fits our case the best

There will be 2 features we will be calculating with Yang-Zhang model:
- `sigma_historical`
- `sigma_today` (Pre NY movement as we don't want to leak NY data into the volatility model)

Yang-Zhang estimator are defined over $n$ trading days where each day $i$ has an overnight gap from the previous daily close $C_{i-1}$ to the open $O_i$:
$$\hat{\sigma}^2_{YZ} = \hat{\sigma}^2_o + k\,\hat{\sigma}^2_c + (1-k)\,\hat{\sigma}^2_{RS}$$

Where:

**Overnight Variance**:

$$\hat{\sigma}^2_o = \frac{1}{n-1} \sum_{i=1}^{n} \left( o_i - \bar{o} \right)^2
\quad \text{where} \quad
o_i = \ln\frac{O_i}{C_{i-1}}, \quad \bar{o} = \frac{1}{n}\sum_{i=1}^{n} o_i$$

**Open to Close Variance**:

$$\hat{\sigma}^2_c = \frac{1}{n-1} \sum_{i=1}^{n} \left( c_i - \bar{c} \right)^2
\quad \text{where} \quad
c_i = \ln\frac{C_i}{O_i}, \quad \bar{c} = \frac{1}{n}\sum_{i=1}^{n} c_i$$

**Rogers Satchell Variance**:

$$\hat{\sigma}^2_{RS} = \frac{1}{n} \sum_{i=1}^{n} \left[
    \ln\frac{H_i}{C_i}\ln\frac{H_i}{O_i} +
    \ln\frac{L_i}{C_i}\ln\frac{L_i}{O_i}
\right]$$

**Optimal Weight**:

$$k = \frac{0.34}{1.34 + \dfrac{n+1}{n-1}}$$

Note: $k$ is derived analytically and is not a tunable parameter, it depends only on $n$

*Limitations: `sigma_today` will underestimates because it only stops at london close since we dont want to leak NY session data, this is a limitation of our approach*
***
### **Sigma Historical (`sigma_historical`)**
Full OHLC including NY. Past days $d < t$ are fully resolved, using $C_{NY}(d)$ for a historical day is not leakage.

$$H^{full}(d) = \max(H_{Asia},\, H_{London},\, H_{NY}), \qquad L^{full}(d) = \min(L_{Asia},\, L_{London},\, L_{NY})$$

**Overnight Variance**:

$$\hat{\sigma}^2_o = \text{RollingVar}_N\!\left(\ln\frac{O_{Asia}(d)}{C_{NY}(d-1)}\right)$$

**Open to Close Variance**:

$$\hat{\sigma}^2_c = \text{RollingVar}_N\!\left(\ln\frac{C_{NY}(d)}{O_{Asia}(d)}\right)$$

**Rogers Satchell Variance**:

$$\hat{\sigma}^2_{RS} = \text{RollingMean}_N\!\left[\ln\frac{H^{full}}{C_{NY}}\ln\frac{H^{full}}{O_{Asia}} + \ln\frac{L^{full}}{C_{NY}}\ln\frac{L^{full}}{O_{Asia}}\right]$$

### **Sigma Today (`sigma_today`)**

Asia + London only, $C_{NY}(t)$ is forbidden.
The OC and RS components are approximated using the last observable price: $C_{London}(t)$.

$$H^{pre}(t) = \max(H_{Asia},\, H_{London}), \qquad L^{pre}(t) = \min(L_{Asia},\, L_{London})$$

**Overnight Variance**:

$$\hat{\sigma}^2_o = \text{RollingVar}_N\!\left(\ln\frac{O_{Asia}(d)}{C_{NY}(d-1)}\right)$$

**Open-to-Close Variance**:

$$\hat{\sigma}^2_{c,proxy} = \text{RollingVar}_N\!\left(\ln\frac{C_{London}(d)}{O_{Asia}(d)}\right)$$

**Rogers-Satchell Variance**:

$$\hat{\sigma}^2_{RS,pre} = \text{RollingMean}_N\!\left[\ln\frac{H^{pre}}{C_{London}}\ln\frac{H^{pre}}{O_{Asia}} + \ln\frac{L^{pre}}{C_{London}}\ln\frac{L^{pre}}{O_{Asia}}\right]$$


In [5]:
# Implementation for Yang Zhang estimator can be seen in `src/ohlc_dss_model/features/volatility.py`
features = yang_zhang(features, HISTORICAL_SPEC)
features = yang_zhang(features, TODAY_SPEC)

print(features.select(["Session", "Sigma_Historical", "Sigma_Today"]).drop_nulls().head(5))

# should be n nulls as it needs n days burn in period
print(f"Null count historical: {features['Sigma_Historical'].null_count()}")
print(f"Null count today: {features['Sigma_Today'].null_count()}")

shape: (5, 3)
┌────────────┬──────────────────┬─────────────┐
│ Session    ┆ Sigma_Historical ┆ Sigma_Today │
│ ---        ┆ ---              ┆ ---         │
│ date       ┆ f64              ┆ f64         │
╞════════════╪══════════════════╪═════════════╡
│ 2016-02-01 ┆ 0.019224         ┆ 0.01157     │
│ 2016-02-02 ┆ 0.019328         ┆ 0.01154     │
│ 2016-02-03 ┆ 0.019555         ┆ 0.011437    │
│ 2016-02-04 ┆ 0.019424         ┆ 0.010962    │
│ 2016-02-05 ┆ 0.019556         ┆ 0.010779    │
└────────────┴──────────────────┴─────────────┘
Null count historical: 20
Null count today: 20


### **Volatility Adjusted Excursion Bands**
***

Intraday price behavior can be simplified into 2 components:
- Adverse Excursion (AE) -> Counter directional movement from reference open $O_{Ref}$
- Favorable Excursion (FE) -> Co directional move following adverse excursion

This framework estimates AE and FE magnitudes based on recent volatility and regime to construct symmetric price bands relative to reference open $O_{Ref}$. This bands are then used to capture Pre NY session interaction for downstream modelling.

***

### **Volatility Scaling**
Excursion prices are normalized using Yang Zhang volatility:
$$\varepsilon^*_{AE}(d) = \frac{\varepsilon_{AE}(d)}{\hat{\sigma}_{YZ}(d-1)}, \qquad \varepsilon^*_{FE}(d) = \frac{\varepsilon_{FE}(d)}{\hat{\sigma}_{YZ}(d-1)}$$

This produce stationary quantities such that it can be used across time.

***

### **Direction Definition**

Normalized daily candle body are defined as such:
$$z_{body}(d) = \frac{|C_{NY}(d) - O_{Asia}(d)}{\hat{\sigma}_{YZ}(d-1)}$$
let $\tau(d)$ be a volatility adaptive threshold:
$$Z_{\sigma}(d) = \frac{\hat{\sigma}_{YZ}(d-1)}{{RollingMean}_{N}(\hat{\sigma}_{YZ})}$$
Hence a day is classified as such:
$${regime}(d) = \begin{cases}
{bullish} \qquad {if }{z_{body}(d) > \tau(d) {and} {C_{NY}(d)} > {O_{Asia}(d)}}\\
{bearish} \qquad {if }{z_{body}(d) > \tau(d) {and} {C_{NY}(d)} < {O_{Asia}(d)}}\\
{neutral} \qquad {if }{z_{body}(d) < \tau(d)}
\end{cases}$$

Neutral here means day represents low volatility days